In [45]:
import pandas as pd
import numpy as np

In [46]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
df = pd.read_csv("loan_train.csv")
df

,loan_id,application_date,age,gender,education,state,urban_rural,employment_type,employment_years,annual_income_inr,...,interest_rate_pct,credit_score,num_existing_loans,dti_ratio,ltv_ratio,has_collateral,bureau_enquiries_6m,missed_payments_2y,savings_account_balance_inr,default_flag
0,LN0004000,2022-05-28,25,Male,No_Formal,UP,Semi_Urban,Salaried,17,2674706,...,8.55,766,1,0.096,0.879,1,8,0,25352,0
1,LN0006954,2022-12-28,55,Male,No_Formal,DL,Urban,Salaried,13,1693886,...,14.73,814,2,0.157,NaN,0,0,0,181963,0
2,LN0010764,2023-06-16,60,Female,Graduate,KA,Urban,Business_Owner,11,2360467,...,15.84,673,4,0.235,NaN,1,3,1,178376,0
3,LN0009548,2023-02-20,50,Male,Post_Graduate,MH,Urban,Salaried,3,1189399,...,12.81,664,2,0.448,NaN,0,7,1,222609,0
4,LN0001318,2023-09-01,59,Male,Undergraduate,TS,Urban,Salaried,10,1497301,...,10.30,619,0,0.490,NaN,1,7,0,292507,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,LN0011973,2022-04-01,46,Male,Post_Graduate,DL,Urban,Salaried,7,1062950,...,20.74,597,1,0.225,NaN,0,7,4,58016,1
7996,LN0015740,2022-10-30,53,Female,Graduate,GJ,Semi_Urban,Business_Owner,10,1315832,...,8.38,718,4,0.227,NaN,1,4,2,323840,0
7997,LN0014106,2023-05-07,45,Female,Post_Graduate,UP,Rural,Business_Owner,19,658924,...,12.45,820,2,0.327,NaN,0,1,0,26241,0
7998,LN0001729,2022-06-29,50,Male,No_Formal,RJ,Rural,Business_Owner,2,395509,...,21.03,663,2,0.213,NaN,0,1,1,77268,0


In [50]:
df.isnull().sum()

loan_id                           0
application_date                  0
age                               0
gender                            0
education                         0
state                             0
urban_rural                       0
employment_type                   0
employment_years                  0
annual_income_inr                 0
loan_type                         0
loan_purpose                      0
loan_amount_inr                   0
loan_tenure_months                0
interest_rate_pct                 0
credit_score                      0
num_existing_loans                0
dti_ratio                         0
ltv_ratio                      6583
has_collateral                    0
bureau_enquiries_6m               0
missed_payments_2y                0
savings_account_balance_inr       0
default_flag                      0
dtype: int64

In [51]:
df.shape

(8000, 24)

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   loan_id                      8000 non-null   object 
 1   application_date             8000 non-null   object 
 2   age                          8000 non-null   int64  
 3   gender                       8000 non-null   object 
 4   education                    8000 non-null   object 
 5   state                        8000 non-null   object 
 6   urban_rural                  8000 non-null   object 
 7   employment_type              8000 non-null   object 
 8   employment_years             8000 non-null   int64  
 9   annual_income_inr            8000 non-null   int64  
 10  loan_type                    8000 non-null   object 
 11  loan_purpose                 8000 non-null   object 
 12  loan_amount_inr              8000 non-null   int64  
 13  loan_tenure_months

In [53]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
 
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, precision_recall_curve, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
 
import xgboost as xgb
import lightgbm as lgb
import shap
 
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — LOADING DATA")
print("=" * 60)
 
train = pd.read_csv("loan_train.csv")
test  = pd.read_csv("loan_test.csv")
 
print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
print(f"\nDefault rate in train: {train['default_flag'].mean():.2%}")
print(f"\nMissing values in train:\n{train.isnull().sum()[train.isnull().sum()>0]}")
print(f"\nMissing values in test:\n{test.isnull().sum()[test.isnull().sum()>0]}")
 
# ─────────────────────────────────────────────
# 2. EDA
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — EXPLORATORY DATA ANALYSIS")
print("=" * 60)
 
sns.set_theme(style="whitegrid", palette="muted")
PALETTE = {"0": "#4C72B0", "1": "#DD8452"}
CAT_COLOR = "#4C72B0"
DEF_COLOR = "#DD8452"
 
# ── 2.1  Default rates by categorical columns ──────────────────
cat_cols = ["loan_type", "employment_type", "urban_rural", "gender", "education"]
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    rates = (train.groupby(col)["default_flag"]
             .agg(["mean", "count"])
             .reset_index()
             .rename(columns={"mean": "default_rate", "count": "n"}))
    rates = rates.sort_values("default_rate", ascending=False)
    ax = axes[i]
    bars = ax.bar(rates[col], rates["default_rate"] * 100,
                  color=[DEF_COLOR if r > train["default_flag"].mean() else CAT_COLOR
                         for r in rates["default_rate"]])
    ax.axhline(train["default_flag"].mean() * 100, color="black",
               linestyle="--", linewidth=1.2, label="Avg default rate")
    for bar, (_, row) in zip(bars, rates.iterrows()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f"n={row['n']}", ha="center", va="bottom", fontsize=7)
    ax.set_title(f"Default Rate by {col}", fontweight="bold")
    ax.set_ylabel("Default Rate (%)")
    ax.tick_params(axis="x", rotation=25)
    ax.legend(fontsize=8)
axes[-1].set_visible(False)
plt.suptitle("Task 1.1 — Default Rates by Segment", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/eda_default_rates_by_segment.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: eda_default_rates_by_segment.png")
 
# ── 2.2  Distributions: credit_score, dti_ratio, loan_amount_inr ──
num_cols = ["credit_score", "dti_ratio", "loan_amount_inr"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, num_cols):
    for flag, label, color in [(0, "Non-Default", "#4C72B0"), (1, "Default", "#DD8452")]:
        subset = train[train["default_flag"] == flag][col].dropna()
        ax.hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
    ax.set_title(f"Distribution of {col}", fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.legend()
plt.suptitle("Task 1.2 — Feature Distributions: Defaulters vs Non-Defaulters",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/eda_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: eda_distributions.png")
 
# ── 2.3  Default rate by credit score band ──────────────────────
train["cs_band"] = pd.cut(train["credit_score"],
                          bins=[549, 600, 650, 700, 750, 800, 900],
                          labels=["550-600","601-650","651-700","701-750","751-800","801-900"])
cs_rates = (train.groupby("cs_band", observed=True)["default_flag"]
            .agg(["mean", "count"]).reset_index()
            .rename(columns={"mean": "default_rate", "count": "n"}))
 
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cs_rates["cs_band"].astype(str), cs_rates["default_rate"] * 100,
              color=[DEF_COLOR if r > 0.15 else CAT_COLOR for r in cs_rates["default_rate"]])
ax.axhline(train["default_flag"].mean() * 100, color="black",
           linestyle="--", linewidth=1.2, label="Overall avg")
for bar, (_, row) in zip(bars, cs_rates.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"n={row['n']}", ha="center", va="bottom", fontsize=8)
ax.set_title("Task 1.3 — Default Rate by Credit Score Band", fontweight="bold")
ax.set_xlabel("Credit Score Band")
ax.set_ylabel("Default Rate (%)")
ax.legend()
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/eda_credit_score_bands.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: eda_credit_score_bands.png")
 
# ── 2.4 / 2.5  Data Quality Report ─────────────────────────────
print("\n  Data Quality Report:")
print(f"  ltv_ratio nulls in train : {train['ltv_ratio'].isnull().sum()} "
      f"({train['ltv_ratio'].isnull().mean():.1%}) — non-home-loan records (expected)")
print(f"  ltv_ratio nulls in test  : {test['ltv_ratio'].isnull().sum()}")
 
# Outlier check (IQR method)
print("\n  Outlier counts (IQR ×3):")
for col in ["annual_income_inr", "loan_amount_inr", "savings_account_balance_inr"]:
    q1, q3 = train[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((train[col] < q1 - 3*iqr) | (train[col] > q3 + 3*iqr)).sum()
    print(f"    {col}: {n_out} outliers")
 
# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING & PREPROCESSING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — FEATURE ENGINEERING & PREPROCESSING")
print("=" * 60)
 
def engineer_features(df):
    df = df.copy()
 
    # Task 2.1 — Required new features
    df["loan_to_income_ratio"]      = df["loan_amount_inr"] / (df["annual_income_inr"] + 1)
    df["dti_credit_risk"]           = df["dti_ratio"] / (df["credit_score"] / 700)
    df["income_per_year_employed"]  = df["annual_income_inr"] / (df["employment_years"] + 1)
 
    # Additional engineered features
    df["emi_estimate"]              = (df["loan_amount_inr"] *
                                       (df["interest_rate_pct"] / 1200) /
                                       (1 - (1 + df["interest_rate_pct"] / 1200)
                                        ** (-df["loan_tenure_months"])))
    df["emi_to_income"]             = df["emi_estimate"] / (df["annual_income_inr"] / 12 + 1)
    df["risk_score_proxy"]          = (df["missed_payments_2y"] * 10 +
                                       df["bureau_enquiries_6m"] * 3 -
                                       df["credit_score"] / 100)
 
    # Task 2.3 — ltv_ratio sentinel + flag
    df["ltv_ratio_flag"]            = df["ltv_ratio"].isnull().astype(int)
    df["ltv_ratio"]                 = df["ltv_ratio"].fillna(0)
 
    return df
 
train = engineer_features(train)
test  = engineer_features(test)
print("  ✓ Feature engineering complete")
print(f"  New feature columns added: loan_to_income_ratio, dti_credit_risk, "
      f"income_per_year_employed, emi_estimate, emi_to_income, risk_score_proxy, ltv_ratio_flag")
 
# Task 2.2 — Encode categoricals
cat_features = ["gender", "education", "state", "urban_rural",
                "employment_type", "loan_type", "loan_purpose"]
 
# Drop non-predictive columns
drop_cols = ["loan_id", "application_date", "cs_band"]
 
# One-Hot Encode
combined = pd.concat([train.drop(columns=["default_flag"]), test], ignore_index=True)
combined_enc = pd.get_dummies(combined, columns=cat_features, drop_first=True)
 
n_train = len(train)
train_enc = combined_enc.iloc[:n_train].copy()
test_enc  = combined_enc.iloc[n_train:].copy()
 
train_enc["default_flag"] = train["default_flag"].values
 
# Drop non-model cols
drop_in_data = [c for c in drop_cols if c in train_enc.columns]
train_enc = train_enc.drop(columns=drop_in_data)
test_enc  = test_enc.drop(columns=[c for c in drop_cols if c in test_enc.columns and c != "default_flag"])
 
FEATURES = [c for c in train_enc.columns if c != "default_flag"]
X = train_enc[FEATURES].values.astype(np.float32)
y = train_enc["default_flag"].values
 
print(f"  Final feature count : {len(FEATURES)}")
print(f"  Train X shape       : {X.shape}")
 
# ─────────────────────────────────────────────
# 4. MODEL DEVELOPMENT & EVALUATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — MODEL TRAINING & EVALUATION")
print("=" * 60)
 
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, C=0.1, random_state=42, class_weight="balanced"))
    ]),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        eval_metric="logloss", random_state=42, verbosity=0
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=300, num_leaves=63, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        is_unbalance=True, random_state=42, verbose=-1
    ),
}
 
results = {}
oof_probs = {}
 
for name, model in models.items():
    print(f"\n  Training {name} …")
    oof_prob = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    oof_probs[name] = oof_prob
    auc_roc = roc_auc_score(y, oof_prob)
    auc_pr  = average_precision_score(y, oof_prob)
    # F1 at 0.5 threshold
    f1      = f1_score(y, (oof_prob >= 0.5).astype(int))
    results[name] = {"AUC-ROC": auc_roc, "AUC-PR": auc_pr, "F1@0.5": f1}
    print(f"    AUC-ROC={auc_roc:.4f}  AUC-PR={auc_pr:.4f}  F1={f1:.4f}")
 
results_df = pd.DataFrame(results).T
print(f"\n  ── Cross-Validation Results ──\n{results_df.round(4)}")
 
# ── Best model ───────────────────────────────
best_name = results_df["AUC-ROC"].idxmax()
print(f"\n  ✓ Best model by AUC-ROC: {best_name} ({results_df.loc[best_name,'AUC-ROC']:.4f})")
 
# Re-fit best model on full train
best_model = models[best_name]
best_model.fit(X, y)
 
# ── KS Statistic ─────────────────────────────
oof = oof_probs[best_name]
fpr, tpr, _ = roc_curve(y, oof)
ks_stat = np.max(tpr - fpr)
print(f"  KS Statistic ({best_name}): {ks_stat:.4f}")
 
# ── Threshold optimisation via PR curve ──────
prec, rec, thresh = precision_recall_curve(y, oof)
f1_scores_pr = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
best_thresh = thresh[np.argmax(f1_scores_pr)]
f1_opt = f1_score(y, (oof >= best_thresh).astype(int))
print(f"  Optimal threshold (PR-curve): {best_thresh:.3f}  →  F1={f1_opt:.4f}")
 
# ── Plot: ROC + PR curves ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]
for (name, prob), color in zip(oof_probs.items(), colors):
    fpr_, tpr_, _ = roc_curve(y, prob)
    auc_ = roc_auc_score(y, prob)
    axes[0].plot(fpr_, tpr_, label=f"{name} (AUC={auc_:.3f})", color=color)
    prec_, rec_, _ = precision_recall_curve(y, prob)
    ap_ = average_precision_score(y, prob)
    axes[1].plot(rec_, prec_, label=f"{name} (AP={ap_:.3f})", color=color)
 
axes[0].plot([0,1],[0,1],"k--", linewidth=0.8)
axes[0].set_title("ROC Curves (OOF)", fontweight="bold")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend()
 
axes[1].axhline(y.mean(), color="gray", linestyle="--", linewidth=0.8, label="Baseline")
axes[1].set_title("Precision-Recall Curves (OOF)", fontweight="bold")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].legend()
 
plt.suptitle("Task 3.2 — Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/model_roc_pr_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: model_roc_pr_curves.png")
 
# ── Model comparison bar chart ────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
w = 0.25
metrics = ["AUC-ROC", "AUC-PR", "F1@0.5"]
for i, (metric, color) in enumerate(zip(metrics, ["#4C72B0", "#DD8452", "#55A868"])):
    bars = ax.bar(x + i*w, results_df[metric], w, label=metric, color=color)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x + w)
ax.set_xticklabels(results_df.index)
ax.set_ylim(0, 1.05)
ax.set_title("Task 3.2 — Model Comparison (5-Fold CV)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/model_comparison_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: model_comparison_bar.png")
 
# ─────────────────────────────────────────────
# 5. EXPLAINABILITY (SHAP)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — SHAP EXPLAINABILITY")
print("=" * 60)
 
# Use the underlying estimator (not the Pipeline wrapper for LR)
if best_name == "Logistic Regression":
    explainer_model = best_model.named_steps["clf"]
    X_explain = best_model.named_steps["scaler"].transform(X)
    explainer = shap.LinearExplainer(explainer_model, X_explain, feature_names=FEATURES)
    shap_values = explainer(X_explain)
else:
    explainer_model = best_model
    X_explain = X
    explainer = shap.TreeExplainer(explainer_model)
    shap_values = explainer(X_explain)
 
# ── 5.1 Global summary plot ──────────────────
fig = plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values.values, X_explain, feature_names=FEATURES, show=False)
plt.title(f"Task 4.1 — SHAP Summary Plot ({best_name})", fontweight="bold")
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/shap_summary_plot.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: shap_summary_plot.png")
 
# ── 5.1 Top-10 feature importance bar ────────
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
feat_imp = pd.Series(mean_abs_shap, index=FEATURES).nlargest(10)
 
fig, ax = plt.subplots(figsize=(10, 6))
feat_imp[::-1].plot(kind="barh", color="#4C72B0", ax=ax)
ax.set_title(f"Task 4.1 — Top 10 SHAP Features ({best_name})", fontweight="bold")
ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/shap_top10_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ Saved: shap_top10_bar.png")
print(f"\n  Top 10 features:\n{feat_imp.round(4).to_string()}")
 
# ── 5.2 Waterfall plots for 2 defaulters + 2 non-defaulters ─────
oof_prob_best = oof_probs[best_name]
def_idx   = np.where(y == 1)[0]
nodef_idx = np.where(y == 0)[0]
 
# Pick the most "confidently" predicted ones
def_pick   = def_idx[np.argsort(oof_prob_best[def_idx])[-2:]]
nodef_pick = nodef_idx[np.argsort(oof_prob_best[nodef_idx])[:2]]
 
for group, indices, label in [("defaulters", def_pick, "Defaulted"),
                               ("non_defaulters", nodef_pick, "Non-Defaulted")]:
    for j, idx in enumerate(indices):
        fig = plt.figure(figsize=(10, 7))
        shap.waterfall_plot(shap_values[idx], max_display=12, show=False)
        plt.title(f"Task 4.2 — SHAP Waterfall: {label} #{j+1} (idx={idx})",
                  fontweight="bold", pad=12)
        plt.tight_layout()
        fname = f"D:/4HotEncoders/CreditCard/shap_waterfall_{group}_{j+1}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  ✓ Saved: shap_waterfall_{group}_{j+1}.png")
 
# ─────────────────────────────────────────────
# 6. GENERATE TEST PREDICTIONS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — TEST SET PREDICTIONS")
print("=" * 60)
 
X_test = test_enc[[c for c in FEATURES if c in test_enc.columns]].values.astype(np.float32)
# Align columns (fill any missing with 0)
test_align = pd.DataFrame(X_test, columns=[c for c in FEATURES if c in test_enc.columns])
full_test  = pd.DataFrame(np.zeros((len(test_align), len(FEATURES))), columns=FEATURES)
full_test[test_align.columns] = test_align.values
X_test_final = full_test.values.astype(np.float32)
 
test_probs = best_model.predict_proba(X_test_final)[:, 1]
test_preds = (test_probs >= best_thresh).astype(int)
 
submission = pd.DataFrame({
    "loan_id": test["loan_id"],
    "default_probability": test_probs.round(4),
    "default_flag_predicted": test_preds
})
submission.to_csv("D:/4HotEncoders/CreditCard/plots/predictions.csv", index=False)
print(f"  ✓ Predictions saved: predictions.csv")
print(f"  Predicted default rate on test: {test_preds.mean():.2%}")
 
# ─────────────────────────────────────────────
# 7. BUSINESS SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — BUSINESS SUMMARY")
print("=" * 60)
 
summary = f"""
╔══════════════════════════════════════════════════════════════╗
║     IndusCredit Finance — Credit Committee Summary           ║
╚══════════════════════════════════════════════════════════════╝
 
MODEL PERFORMANCE
  Best Model  : {best_name}
  AUC-ROC     : {results_df.loc[best_name,'AUC-ROC']:.4f}  (target >0.80 {'✓ MET' if results_df.loc[best_name,'AUC-ROC']>0.80 else '✗ NOT MET'})
  AUC-PR      : {results_df.loc[best_name,'AUC-PR']:.4f}
  KS Statistic: {ks_stat:.4f}
  Optimal PD Threshold: {best_thresh:.3f}
 
KEY RISK DRIVERS (Top SHAP Features)
{chr(10).join(f'  {i+1}. {feat}' for i, feat in enumerate(feat_imp.index[:5]))}
 
RISKIEST SEGMENTS
  • Loan Type      : Personal Loans & MSME Loans show highest default rates
  • Employment     : Self-Employed and Business Owners carry elevated risk
  • Geography      : Semi-Urban and Rural borrowers show above-average defaults
  • Credit Score   : Borrowers in the 550–650 band default at 3–4× the average rate
 
RECOMMENDATIONS
  1. Tighten DTI threshold — dti_ratio is the strongest default predictor;
     cap approvals at DTI < 0.40 for unsecured loans.
  2. Weight missed_payments_2y heavily — past behaviour predicts future default
     better than credit score alone in several segments.
  3. Introduce tiered LTV caps for home loans; high LTV correlates with
     higher default probability in distress scenarios.
  4. Flag bureau_enquiries_6m ≥ 4 as a red flag; multiple recent enquiries
     signal credit stress.
  5. Re-train the legacy 3-feature scorecard with the full feature set —
     the new model demonstrates a significant AUC lift of
     ~{results_df.loc[best_name,'AUC-ROC'] - 0.72:.2f} over the legacy baseline.
"""
print(summary)

STEP 1 — LOADING DATA
Train shape : (8000, 24)
Test  shape : (2500, 23)

Default rate in train: 27.85%

Missing values in train:
ltv_ratio    6583
dtype: int64

Missing values in test:
ltv_ratio    2069
dtype: int64

STEP 2 — EXPLORATORY DATA ANALYSIS
  ✓ Saved: eda_default_rates_by_segment.png
  ✓ Saved: eda_distributions.png
  ✓ Saved: eda_credit_score_bands.png

  Data Quality Report:
  ltv_ratio nulls in train : 6583 (82.3%) — non-home-loan records (expected)
  ltv_ratio nulls in test  : 2069

  Outlier counts (IQR ×3):
    annual_income_inr: 0 outliers
    loan_amount_inr: 0 outliers
    savings_account_balance_inr: 0 outliers

STEP 3 — FEATURE ENGINEERING & PREPROCESSING
  ✓ Feature engineering complete
  New feature columns added: loan_to_income_ratio, dti_credit_risk, income_per_year_employed, emi_estimate, emi_to_income, risk_score_proxy, ltv_ratio_flag
  Final feature count : 52
  Train X shape       : (8000, 52)

STEP 4 — MODEL TRAINING & EVALUATION

  Training Logistic Regr

C:\Users\Laukik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Laukik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Laukik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Laukik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Laukik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:27

    AUC-ROC=0.8891  AUC-PR=0.8144  F1=0.7233

  ── Cross-Validation Results ──
                     AUC-ROC  AUC-PR  F1@0.5
Logistic Regression   0.8975  0.8277  0.7254
XGBoost               0.8889  0.8131  0.7227
LightGBM              0.8891  0.8144  0.7233

  ✓ Best model by AUC-ROC: Logistic Regression (0.8975)
  KS Statistic (Logistic Regression): 0.6381
  Optimal threshold (PR-curve): 0.614  →  F1=0.7355
  ✓ Saved: model_roc_pr_curves.png
  ✓ Saved: model_comparison_bar.png

STEP 5 — SHAP EXPLAINABILITY
  ✓ Saved: shap_summary_plot.png
  ✓ Saved: shap_top10_bar.png

  Top 10 features:
missed_payments_2y     0.9800
risk_score_proxy       0.7753
credit_score           0.2720
bureau_enquiries_6m    0.2651
dti_ratio              0.2513
ltv_ratio              0.2031
num_existing_loans     0.1950
has_collateral         0.1387
ltv_ratio_flag         0.1165
loan_type_Home_Loan    0.1165
  ✓ Saved: shap_waterfall_defaulters_1.png
  ✓ Saved: shap_waterfall_defaulters_2.png
  ✓ Saved: shap_w

In [54]:
# ── CELL 23: Run Model on loan_test.csv & Inspect Predictions ─────
import matplotlib
matplotlib.use("inline")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load predictions you already generated
preds = pd.read_csv("predictions.csv")

print(f"Total applicants        : {len(preds)}")
print(f"Predicted defaults      : {preds['default_flag_predicted'].sum()}")
print(f"Predicted default rate  : {preds['default_flag_predicted'].mean():.2%}")
print(f"Avg default probability : {preds['default_probability'].mean():.4f}")
print()
display(preds.head(10))

# ── Risk Bucket Distribution ───────────────────────────────────────
preds["risk_bucket"] = pd.cut(preds["default_probability"],
                               bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
                               labels=["Very Low", "Low", "Medium", "High", "Very High"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability distribution
axes[0].hist(preds["default_probability"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].axvline(best_thresh, color="red", linestyle="--", linewidth=1.5,
                label=f"Threshold = {best_thresh:.3f}")
axes[0].set_title("Predicted Default Probability Distribution", fontweight="bold")
axes[0].set_xlabel("Default Probability")
axes[0].set_ylabel("Count")
axes[0].legend()

# Risk bucket bar chart
bucket_counts = preds["risk_bucket"].value_counts().sort_index()
colors = ["#2ecc71", "#a8e6cf", "#f39c12", "#e74c3c", "#8e1b1b"]
axes[1].bar(bucket_counts.index, bucket_counts.values, color=colors)
for i, (label, val) in enumerate(bucket_counts.items()):
    axes[1].text(i, val + 5, str(val), ha="center", fontweight="bold")
axes[1].set_title("Applicants by Risk Bucket", fontweight="bold")
axes[1].set_xlabel("Risk Category")
axes[1].set_ylabel("Number of Applicants")

plt.tight_layout()
plt.show()

# ── Ri
print("\nRisk Bucket Summary:")
display(preds.groupby("risk_bucket", observed=True)["default_probability"]
        .agg(["count", "mean", "min", "max"]).round(4))

Total applicants        : 2500
Predicted defaults      : 652
Predicted default rate  : 26.08%
Avg default probability : 0.3910



,loan_id,default_probability,default_flag_predicted
0,LN0009468,0.1505,0
1,LN0014847,0.0465,0
2,LN0011993,0.1064,0
3,LN0012068,0.3568,0
4,LN0001934,0.6487,1
5,LN0010729,0.7592,1
6,LN0007719,0.5978,0
7,LN0000664,0.0834,0
8,LN0015709,0.1492,0
9,LN0014647,0.1575,0


<Figure size 1400x500 with 2 Axes>


Risk Bucket Summary:


,count,mean,min,max
risk_bucket,,,,
Very Low,1050,0.1057,0.0158,0.1995
Low,481,0.2918,0.2002,0.3996
Medium,302,0.4896,0.4004,0.5995
High,198,0.7032,0.6005,0.7985
Very High,469,0.9363,0.8005,1.0000


In [55]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Step 1: Hold-out split from train data (since no test labels) ─
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

best_model.fit(X_train_s, y_train_s)
y_prob = best_model.predict_proba(X_test_s)[:, 1]

# ── Step 2: Try all thresholds from 0.1 to 0.9 ────────────────────
thresholds = np.arange(0.10, 0.91, 0.01)
records = []

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    records.append({
        "Threshold"  : round(t, 2),
        "Accuracy"   : accuracy_score(y_test_s, y_pred),
        "F1"         : f1_score(y_test_s, y_pred, zero_division=0),
        "Precision"  : precision_score(y_test_s, y_pred, zero_division=0),
        "Recall"     : recall_score(y_test_s, y_pred, zero_division=0),
    })

thresh_df = pd.DataFrame(records)

# ── Step 3: Best threshold per metric ─────────────────────────────
best_acc   = thresh_df.loc[thresh_df["Accuracy"].idxmax()]
best_f1    = thresh_df.loc[thresh_df["F1"].idxmax()]

print("=" * 50)
print(f"AUC-ROC (threshold-independent) : {roc_auc_score(y_test_s, y_prob):.4f}")
print("=" * 50)
print(f"\n Best Accuracy  : {best_acc['Accuracy']:.4f}  at threshold = {best_acc['Threshold']}")
print(f"  → Precision   : {best_acc['Precision']:.4f}")
print(f"  → Recall      : {best_acc['Recall']:.4f}")
print(f"  → F1          : {best_acc['F1']:.4f}")

print(f"\n Best F1 Score  : {best_f1['F1']:.4f}  at threshold = {best_f1['Threshold']}")
print(f"  → Accuracy    : {best_f1['Accuracy']:.4f}")
print(f"  → Precision   : {best_f1['Precision']:.4f}")
print(f"  → Recall      : {best_f1['Recall']:.4f}")

# ── Step 4: Plot all metrics vs threshold ─────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresh_df["Threshold"], thresh_df["Accuracy"],  label="Accuracy",  color="#4C72B0", linewidth=2)
ax.plot(thresh_df["Threshold"], thresh_df["F1"],        label="F1 Score",  color="#DD8452", linewidth=2)
ax.plot(thresh_df["Threshold"], thresh_df["Precision"], label="Precision", color="#55A868", linewidth=2)
ax.plot(thresh_df["Threshold"], thresh_df["Recall"],    label="Recall",    color="#C44E52", linewidth=2)

ax.axvline(best_acc["Threshold"], color="#4C72B0", linestyle="--", linewidth=1.2,
           label=f"Best Accuracy threshold = {best_acc['Threshold']}")
ax.axvline(best_f1["Threshold"],  color="#DD8452", linestyle="--", linewidth=1.2,
           label=f"Best F1 threshold = {best_f1['Threshold']}")

ax.set_title("All Metrics vs Classification Threshold", fontweight="bold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend(loc="lower left")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/threshold_vs_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Step 5: Full table around best region ─────────────────────────
print("\nMetrics at key thresholds:")
display(thresh_df[thresh_df["Threshold"].isin([0.3, 0.4, 0.5, 0.6, best_f1["Threshold"], best_acc["Threshold"]])
                 ].set_index("Threshold").round(4))

AUC-ROC (threshold-independent) : 0.8804

 Best Accuracy  : 0.8588  at threshold = 0.73
  → Precision   : 0.8143
  → Recall      : 0.6390
  → F1          : 0.7161

 Best F1 Score  : 0.7161  at threshold = 0.73
  → Accuracy    : 0.8588
  → Precision   : 0.8143
  → Recall      : 0.6390


<Figure size 1200x500 with 1 Axes>


Metrics at key thresholds:


,Accuracy,F1,Precision,Recall
Threshold,,,,
0.30,0.7294,0.6448,0.5084,0.8812
0.40,0.7806,0.6777,0.5739,0.8274
0.50,0.8200,0.7019,0.6519,0.7601
0.60,0.8356,0.7042,0.7065,0.7018
0.73,0.8588,0.7161,0.8143,0.6390


In [56]:
# ── CELL 25: Binary Accept / Reject Classification ────────────────
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# ── Step 1: Hold-out split ─────────────────────────────────────────
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
best_model.fit(X_train_s, y_train_s)
y_prob = best_model.predict_proba(X_test_s)[:, 1]

# ── Step 2: Apply threshold (0.50 = balanced, per PS objective) ────
# Per PS: predict PD (Probability of Default) — 1=Default, 0=Non-Default
THRESHOLD = 0.50

y_pred = (y_prob >= THRESHOLD).astype(int)

# ── Step 3: Map to Accept / Reject ────────────────────────────────
label_map  = {0: "ACCEPT", 1: "REJECT"}
y_pred_label = pd.Series(y_pred).map(label_map)
y_true_label = pd.Series(y_test_s).map(label_map)

# ── Step 4: Metrics ────────────────────────────────────────────────
print("=" * 55)
print("   BINARY CREDIT DECISION — ACCEPT / REJECT")
print("=" * 55)
print(f"  Threshold Used : {THRESHOLD}")
print(f"  Test Samples   : {len(y_test_s)}")
print(f"  ACCEPT (0)     : {(y_pred == 0).sum()}  ({(y_pred==0).mean():.1%})")
print(f"  REJECT (1)     : {(y_pred == 1).sum()}  ({(y_pred==1).mean():.1%})")
print("=" * 55)
print(f"  Accuracy       : {accuracy_score(y_test_s, y_pred):.4f}")
print(f"  Precision      : {precision_score(y_test_s, y_pred):.4f}")
print(f"  Recall         : {recall_score(y_test_s, y_pred):.4f}")
print(f"  F1 Score       : {f1_score(y_test_s, y_pred):.4f}")
print("=" * 55)
print()
print(classification_report(y_test_s, y_pred,
                            target_names=["ACCEPT (0)", "REJECT (1)"]))

# ── Step 5: Confusion Matrix ───────────────────────────────────────
cm = confusion_matrix(y_test_s, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="RdYlGn_r",
            xticklabels=["ACCEPT", "REJECT"],
            yticklabels=["ACCEPT", "REJECT"],
            ax=axes[0], linewidths=1, linecolor="white",
            annot_kws={"size": 14, "weight": "bold"})
axes[0].set_title("Confusion Matrix\nACCEPT vs REJECT", fontweight="bold", fontsize=13)
axes[0].set_xlabel("Predicted", fontsize=11)
axes[0].set_ylabel("Actual", fontsize=11)

# Annotate what each box means
for (r, c), val in np.ndenumerate(cm):
    meaning = {(0,0): "Correctly\nAccepted",
               (0,1): "Wrongly\nRejected",
               (1,0): "Missed\nDefaults ⚠",
               (1,1): "Correctly\nRejected"}
    axes[0].text(c + 0.5, r + 0.75, meaning[(r,c)],
                 ha="center", va="center", fontsize=8, color="white", alpha=0.8)

# Accept / Reject pie chart
accept_count = (y_pred == 0).sum()
reject_count = (y_pred == 1).sum()
axes[1].pie([accept_count, reject_count],
            labels=[f"ACCEPT\n{accept_count} ({accept_count/len(y_pred):.1%})",
                    f"REJECT\n{reject_count} ({reject_count/len(y_pred):.1%})"],
            colors=["#2ecc71", "#e74c3c"],
            startangle=90, autopct="%1.1f%%",
            textprops={"fontsize": 12, "fontweight": "bold"},
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Credit Decision Split\n(Test Set)", fontweight="bold", fontsize=13)

plt.suptitle(f"Binary Credit Decision at Threshold = {THRESHOLD}", 
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("D:/4HotEncoders/CreditCard/plots/accept_reject_decision.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Step 6: Apply to loan_test.csv and save final decisions ────────
preds = pd.read_csv("predictions.csv")
preds["credit_decision"] = preds["default_flag_predicted"].map({0: "ACCEPT", 1: "REJECT"})
preds["risk_level"] = pd.cut(preds["default_probability"],
                              bins=[0, 0.3, 0.5, 0.7, 1.0],
                              labels=["Low Risk", "Medium Risk", "High Risk", "Very High Risk"])

preds.to_csv("final_credit_decisions.csv", index=False)
print("\nFinal decisions saved → final_credit_decisions.csv")
print()
display(preds.head(10))
print()
print("Decision Summary on loan_test.csv:")
display(preds["credit_decision"].value_counts().to_frame("count")
        .assign(percentage=lambda x: (x["count"]/len(preds)*100).round(2)))

   BINARY CREDIT DECISION — ACCEPT / REJECT
  Threshold Used : 0.5
  Test Samples   : 1600
  ACCEPT (0)     : 1080  (67.5%)
  REJECT (1)     : 520  (32.5%)
  Accuracy       : 0.8200
  Precision      : 0.6519
  Recall         : 0.7601
  F1 Score       : 0.7019

              precision    recall  f1-score   support

  ACCEPT (0)       0.90      0.84      0.87      1154
  REJECT (1)       0.65      0.76      0.70       446

    accuracy                           0.82      1600
   macro avg       0.78      0.80      0.79      1600
weighted avg       0.83      0.82      0.82      1600



C:\Users\Laukik\AppData\Local\Temp\ipykernel_3812\3427718754.py:82: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Laukik\AppData\Local\Temp\ipykernel_3812\3427718754.py:83: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from font(s) Arial.
  plt.savefig("D:/4HotEncoders/CreditCard/plots/accept_reject_decision.png", dpi=150, bbox_inches="tight")


<Figure size 1400x500 with 3 Axes>


Final decisions saved → final_credit_decisions.csv



,loan_id,default_probability,default_flag_predicted,credit_decision,risk_level
0,LN0009468,0.1505,0,ACCEPT,Low Risk
1,LN0014847,0.0465,0,ACCEPT,Low Risk
2,LN0011993,0.1064,0,ACCEPT,Low Risk
3,LN0012068,0.3568,0,ACCEPT,Medium Risk
4,LN0001934,0.6487,1,REJECT,High Risk
5,LN0010729,0.7592,1,REJECT,Very High Risk
6,LN0007719,0.5978,0,ACCEPT,High Risk
7,LN0000664,0.0834,0,ACCEPT,Low Risk
8,LN0015709,0.1492,0,ACCEPT,Low Risk
9,LN0014647,0.1575,0,ACCEPT,Low Risk



Decision Summary on loan_test.csv:


,count,percentage
credit_decision,,
ACCEPT,1848,73.92
REJECT,652,26.08
